## 1. Architecture Overview (ETL + Storage Design)

### High-Level Flow
1. **Raw ingestion** (chunked read for scalability)
2. **Cleaning + validation** (data quality rules)
3. **Transformation** (canonical schema + derived fields)
4. **Load to PostgreSQL** (typed fact table + indexes)
5. **Serve analytics** (SQL queries + optional aggregates)

### Tables
- `dim_taxi_zones` (Zone metadata)
- `fact_taxi_trips` (Trip events)

### Notes
- Documented a production-style ETL architecture.
- Split storage into `dim_` and `fact_` tables for analytics.
- Clarified why indexing + validation matter in real pipelines.

## 2. Environment Setup & Dependencies

This notebook uses:
- **pandas** for processing (chunking for CSV)
- **SQLAlchemy** for database connectivity
- **PostgreSQL** as the analytics store

Optional:
- **Prefect** or **Airflow** for orchestration (conceptual in this repo)

In [2]:
import os
import warnings
from typing import Dict, Optional, Tuple

import pandas as pd
import pyarrow
from sqlalchemy import create_engine, text
from sqlalchemy.engine import Engine

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


### Notes
- Loaded core libraries for chunked ETL (pandas) and database IO (SQLAlchemy).
- Enabled readable notebook outputs for audits and QA checks.

## 3. Data Source Description (NYC TLC)

### Official / Canonical Links
- TLC Trip Record Data page: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page  
- Yellow Taxi data dictionary (PDF): https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf  
- Taxi Zone Lookup (CSV): https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv  


In [3]:
# Environment & configuration
import os
from pathlib import Path

# Load .env (if present) from the notebook's working directory
try:
    from dotenv import load_dotenv
    env_path = Path.cwd() / ".env"
    if env_path.exists():
        load_dotenv(dotenv_path=env_path)
    else:
        print(f"Note: .env not found at {env_path} — using system environment variables.")
except ModuleNotFoundError:
    print("Note: python-dotenv is not installed — using system environment variables only.")

PG_USER = os.getenv("PG_USER")
PG_PASSWORD = os.getenv("PG_PASSWORD")
PG_HOST = os.getenv("PG_HOST", "localhost")
PG_PORT = os.getenv("PG_PORT", "5432")
PG_DB = os.getenv("PG_DB")

assert all([PG_USER, PG_PASSWORD, PG_DB]), (
    "Database environment variables not set.\n"
    "Set PG_USER, PG_PASSWORD, and PG_DB in a .env file (recommended) or in your system environment.\n"
    "If using a .env file, place it in the notebook working directory (print(os.getcwd()))."
)

DATABASE_URL = f"postgresql+psycopg2://{PG_USER}:{PG_PASSWORD}@{PG_HOST}:{PG_PORT}/{PG_DB}"

# Data inputs (override via environment variables if you want a different month)
TRIPS_MONTH_URL = os.getenv(
    "TRIPS_MONTH_URL",
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
)

DATA_DIR = "./data"
os.makedirs(DATA_DIR, exist_ok=True)
RAW_TRIPS_PATH = os.path.join(DATA_DIR, "yellow_tripdata.parquet")

print(f"Database configured: {PG_USER}@{PG_HOST}:{PG_PORT}/{PG_DB}")
print(f"Trip source set: {TRIPS_MONTH_URL}")
print(f"Local trip path: {RAW_TRIPS_PATH}")


Connected to database: postgres@localhost:5432/taxi_db


## Preflight checks

This cell checks that optional dependencies are available and that your configuration is set. It does **not** download data or connect to the database.


In [4]:
import importlib

def _has(pkg: str) -> bool:
    try:
        importlib.import_module(pkg)
        return True
    except Exception:
        return False

print('Parquet engine available (pyarrow):', _has('pyarrow'))
print('Postgres driver available (psycopg2):', _has('psycopg2'))
print('TRIPS_MONTH_URL set:', bool(TRIPS_MONTH_URL))
print('Expected trip file path:', RAW_TRIPS_PATH)


Parquet engine available (pyarrow): True
Postgres driver available (psycopg2): True
TRIPS_MONTH_URL set: True
Expected trip file path: ./data\yellow_tripdata.parquet


### Notes
- Set up reproducible paths (`./data`) and Postgres config via environment variables.
- Reserved `TRIPS_MONTH_URL` for a monthly Yellow Taxi download link from the official TLC page.

**Default for this notebook:** Yellow Taxi January 2023 (`yellow_tripdata_2023-01.parquet`).


## 4. Download Helpers (Trip file + Zone lookup)

This cell downloads:
- `taxi_zone_lookup.csv` (stable link)
- a monthly Yellow Taxi file (you provide `TRIPS_MONTH_URL`)

If you downloaded files manually, just place them in `./data/` and skip download.

**Default for this notebook:** Yellow Taxi — January 2023 (`yellow_tripdata_2023-01.parquet`).


In [5]:
import urllib.request
import shutil
from urllib.error import URLError, HTTPError

def _looks_like_parquet(path: str) -> bool:
    """
    Minimal Parquet sanity check:
    - Parquet files start with b'PAR1' and end with b'PAR1' (footer magic)
    """
    try:
        with open(path, "rb") as f:
            head = f.read(4)
            if head != b"PAR1":
                return False
            f.seek(-4, os.SEEK_END)
            tail = f.read(4)
            return tail == b"PAR1"
    except Exception:
        return False

def _looks_like_csv(path: str) -> bool:
    """
    Minimal CSV sanity check:
    - File begins with printable text and contains commas/newlines
    """
    try:
        with open(path, "rb") as f:
            head = f.read(200)
        if b"\x00" in head:
            return False
        return (b"," in head) and (b"\n" in head or b"\r\n" in head)
    except Exception:
        return False

def download_file(url: str, dst_path: str, timeout: int = 60, expected=None) -> None:
    """
    Download a file robustly:
    - expected: "parquet" or "csv" (optional). If provided, validates the downloaded file.
    - Uses temp file + atomic rename
    - Re-downloads if an existing file is present but fails validation
    """
    if not url:
        raise ValueError("Download URL is empty. Provide a valid URL.")
    dst_path = os.path.normpath(dst_path)
    tmp_path = dst_path + ".part"

    def _is_valid(p: str) -> bool:
        if expected == "parquet":
            return _looks_like_parquet(p)
        if expected == "csv":
            return _looks_like_csv(p)
        return True

    if os.path.exists(dst_path) and os.path.getsize(dst_path) > 0:
        if _is_valid(dst_path):
            print(f"Already exists: {dst_path}")
            return
        else:
            print(f"Existing file failed validation — deleting: {dst_path}")
            try:
                os.remove(dst_path)
            except Exception:
                pass

    print(f"Downloading → {dst_path}")
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=timeout) as r, open(tmp_path, "wb") as f:
            shutil.copyfileobj(r, f)

        if not _is_valid(tmp_path):
            try:
                with open(tmp_path, "rb") as f:
                    preview = f.read(200)
            except Exception:
                preview = b""
            try:
                os.remove(tmp_path)
            except Exception:
                pass
            raise RuntimeError(
                "Downloaded file failed validation. "
                "This usually happens if the download was blocked/interrupted or the URL returned an unexpected file type. "
                f"Expected={expected!r}. Preview(first bytes)={preview!r}"
            )

        os.replace(tmp_path, dst_path)
        print("Done.")
    except (HTTPError, URLError, TimeoutError) as e:
        if os.path.exists(tmp_path):
            try:
                os.remove(tmp_path)
            except Exception:
                pass
        raise RuntimeError(f"Download failed: {e}")
    if os.path.exists(dst_path) and os.path.getsize(dst_path) > 0:
        print(f"Already exists: {dst_path}")
        return
    print(f"Downloading → {dst_path}")
    download_file(url, dst_path)  # handled above
    print("Done.")

ZONE_LOOKUP_URL = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
ZONE_LOOKUP_PATH = os.path.join(DATA_DIR, "taxi_zone_lookup.csv")

print("Using monthly trips URL:", TRIPS_MONTH_URL)
download_file(TRIPS_MONTH_URL, RAW_TRIPS_PATH, expected='parquet')

download_file(ZONE_LOOKUP_URL, ZONE_LOOKUP_PATH, expected='csv')


Using monthly trips URL: https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet
Already exists: data\yellow_tripdata.parquet
Already exists: data\taxi_zone_lookup.csv


### Notes
- Downloaded the taxi zone lookup CSV.
- Prepared a clean mechanism to download a chosen monthly TLC file via `TRIPS_MONTH_URL`.
- Kept URLs configurable to keep the repo resilient to source path changes.

## 5. Load Zone Dimension (`dim_taxi_zones`)

We load taxi zone metadata so trip records can be enriched from IDs to borough/zone names.
This is a strong : **dimensional modeling + analytics usability**.

In [6]:
zones = pd.read_csv(ZONE_LOOKUP_PATH)
zones.head(), zones.shape, zones.columns.tolist()


(   LocationID        Borough                     Zone service_zone
 0           1            EWR           Newark Airport          EWR
 1           2         Queens              Jamaica Bay    Boro Zone
 2           3          Bronx  Allerton/Pelham Gardens    Boro Zone
 3           4      Manhattan            Alphabet City  Yellow Zone
 4           5  Staten Island            Arden Heights    Boro Zone,
 (265, 4),
 ['LocationID', 'Borough', 'Zone', 'service_zone'])

### Notes
- Loaded taxi zone metadata successfully.
- Confirmed the dataset contains LocationID and descriptors for enrichment.

## 6. ETL Functions (Extract → Transform → Load)

Design principles:
- **Defensive**: type coercion, missing checks
- **Scalable**: chunking for CSV
- **Auditable**: per-load drop statistics
- **Analytics-ready**: canonical schema + derived time fields

In [7]:
YELLOW_PICKUP = "tpep_pickup_datetime"
YELLOW_DROPOFF = "tpep_dropoff_datetime"

CANONICAL_COLS = [
    "VendorID",
    YELLOW_PICKUP,
    YELLOW_DROPOFF,
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "total_amount",
]

def coerce_trip_types(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Datetimes
    for c in [YELLOW_PICKUP, YELLOW_DROPOFF]:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], errors="coerce")

    # Numerics
    for c in ["passenger_count", "trip_distance", "fare_amount", "total_amount",
              "PULocationID", "DOLocationID", "payment_type", "VendorID"]:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    return df

def clean_validate(df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[str, int]]:
    stats = {"rows_in": len(df)}
    df = df.copy()

    # Keep only columns we care about (if present)
    keep = [c for c in CANONICAL_COLS if c in df.columns]
    df = df[keep].copy()

    df = coerce_trip_types(df)

    # Drop rows missing essentials
    required = [YELLOW_PICKUP, YELLOW_DROPOFF, "trip_distance", "total_amount"]
    required = [c for c in required if c in df.columns]
    before = len(df)
    df = df.dropna(subset=required)
    stats["dropped_missing_required"] = before - len(df)

    # Validation rules
    before = len(df)
    df = df[df["trip_distance"] > 0]
    stats["dropped_nonpositive_distance"] = before - len(df)

    if "fare_amount" in df.columns:
        before = len(df)
        df = df[df["fare_amount"] >= 0]
        stats["dropped_negative_fare"] = before - len(df)

    before = len(df)
    df = df[df["total_amount"] >= 0]
    stats["dropped_negative_total"] = before - len(df)

    # Time order
    before = len(df)
    df = df[df[YELLOW_DROPOFF] >= df[YELLOW_PICKUP]]
    stats["dropped_bad_time_order"] = before - len(df)

    # Derived fields
    df["trip_duration_min"] = (df[YELLOW_DROPOFF] - df[YELLOW_PICKUP]).dt.total_seconds() / 60.0
    df["pickup_date"] = df[YELLOW_PICKUP].dt.date
    df["pickup_hour"] = df[YELLOW_PICKUP].dt.hour

    before = len(df)
    df = df[(df["trip_duration_min"] > 0) & (df["trip_duration_min"] <= 24 * 60)]
    stats["dropped_absurd_duration"] = before - len(df)

    stats["rows_out"] = len(df)
    return df, stats


### Notes
- Implemented canonical schema selection, robust type coercion, and validation rules.
- Added analytics-ready columns: `trip_duration_min`, `pickup_date`, `pickup_hour`.
- Produced drop statistics to quantify data-quality impact.

## 7. Database Design + Initialization (PostgreSQL)

We create:
- `dim_taxi_zones` (small dimension)
- `fact_taxi_trips` (large fact table)

Indexes support:
- time-series filtering
- daily rollups
- zone-based demand queries

In [8]:
def get_engine() -> Engine:
    return create_engine(DATABASE_URL, pool_pre_ping=True)

engine = get_engine()

CREATE_SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS dim_taxi_zones (
  location_id INTEGER PRIMARY KEY,
  borough     TEXT,
  zone        TEXT,
  service_zone TEXT
);

CREATE TABLE IF NOT EXISTS fact_taxi_trips (
  vendor_id         INTEGER,
  pickup_datetime   TIMESTAMP,
  dropoff_datetime  TIMESTAMP,
  passenger_count   DOUBLE PRECISION,
  trip_distance     DOUBLE PRECISION,
  pu_location_id    INTEGER,
  do_location_id    INTEGER,
  payment_type      INTEGER,
  fare_amount       DOUBLE PRECISION,
  total_amount      DOUBLE PRECISION,
  trip_duration_min DOUBLE PRECISION,
  pickup_date       DATE,
  pickup_hour       INTEGER
);

CREATE INDEX IF NOT EXISTS idx_trips_pickup_dt   ON fact_taxi_trips(pickup_datetime);
CREATE INDEX IF NOT EXISTS idx_trips_pickup_date ON fact_taxi_trips(pickup_date);
CREATE INDEX IF NOT EXISTS idx_trips_pu_date     ON fact_taxi_trips(pu_location_id, pickup_date);
CREATE INDEX IF NOT EXISTS idx_trips_do_date     ON fact_taxi_trips(do_location_id, pickup_date);
"""

with engine.begin() as conn:
    conn.execute(text(CREATE_SCHEMA_SQL))

print("Schema initialized.")


Schema initialized.


### Notes
- Created `dim_taxi_zones` + `fact_taxi_trips` as an analytics-friendly model.
- Added indexes aligned to time-series and zone demand queries.
- Prepared the database for high-volume append loads.

## 8. Load `dim_taxi_zones`

Dimension tables should be small, clean, and easy to refresh (idempotent loads).

In [9]:
zones_db = zones.rename(columns={
    "LocationID": "location_id",
    "Borough": "borough",
    "Zone": "zone",
    "service_zone": "service_zone",
    "ServiceZone": "service_zone",
}).copy()

needed = ["location_id", "borough", "zone", "service_zone"]
zones_db = zones_db[[c for c in needed if c in zones_db.columns]].dropna(subset=["location_id"])
zones_db["location_id"] = pd.to_numeric(zones_db["location_id"], errors="coerce").astype("Int64")

with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE dim_taxi_zones;"))

zones_db.to_sql("dim_taxi_zones", engine, if_exists="append", index=False, method="multi", chunksize=5000)
zones_db.head()


,location_id,borough,zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


### Notes
- Loaded taxi zone lookup data into `dim_taxi_zones`.
- Used TRUNCATE + append for a clean, repeatable dimension refresh.
- Zone IDs are now available for enriching trip facts.

## 9. Load Trip Facts (`fact_taxi_trips`) — Parquet or CSV

- **Parquet**: fast read, stable typing (recommended).
- **CSV**: use chunking to avoid memory blowups.

Set `WIPE_FACT=true` for repeatable demo runs.

In [10]:
def write_fact(df: pd.DataFrame) -> int:
    if df.empty:
        return 0

    out = df.rename(columns={
        "VendorID": "vendor_id",
        YELLOW_PICKUP: "pickup_datetime",
        YELLOW_DROPOFF: "dropoff_datetime",
        "PULocationID": "pu_location_id",
        "DOLocationID": "do_location_id",
    }).copy()

    cols = [
        "vendor_id","pickup_datetime","dropoff_datetime","passenger_count","trip_distance",
        "pu_location_id","do_location_id","payment_type","fare_amount","total_amount",
        "trip_duration_min","pickup_date","pickup_hour"
    ]
    out = out[[c for c in cols if c in out.columns]]

    out.to_sql("fact_taxi_trips", engine, if_exists="append", index=False, method="multi", chunksize=50_000)
    return len(out)

def load_trips(path: str) -> pd.DataFrame:
    """
    Load NYC TLC trip data into PostgreSQL.
    Supports Parquet (preferred) or CSV (chunked).
    """
    ext = os.path.splitext(path)[1].lower()

    wipe_fact = os.getenv("WIPE_FACT", "true").lower() == "true"
    if wipe_fact:
        with engine.begin() as conn:
            conn.execute(text("TRUNCATE TABLE fact_taxi_trips;"))
        print("fact_taxi_trips truncated (repeatable run).")

    if ext == ".parquet":
        try:
            df = pd.read_parquet(path)
        except Exception as e:
            raise RuntimeError("Failed to read Parquet file. If you see \"Parquet magic bytes not found\", the file is usually corrupted or was downloaded as an HTML error page. Delete ./data/yellow_tripdata.parquet and re-run the download cell. Ensure TRIPS_MONTH_URL points to a .parquet file and your internet is not blocking CloudFront. Original error: " + str(e))

        clean, stats = clean_validate(df)
        loaded = write_fact(clean)
        return pd.DataFrame([{**stats, "rows_loaded": loaded, "mode": "parquet"}])

    audits = []
    for i, chunk in enumerate(pd.read_csv(path, chunksize=CHUNKSIZE), start=1):
        clean, stats = clean_validate(chunk)
        loaded = write_fact(clean)
        audits.append({**stats, "rows_loaded": loaded, "chunk": i, "mode": "csv"})
        print(
            f"[chunk {i}] in={stats['rows_in']:,} "
            f"out={stats['rows_out']:,} "
            f"loaded={loaded:,}"
        )

    return pd.DataFrame(audits)

# Ensure the trip file exists (download if needed)
if not os.path.exists(RAW_TRIPS_PATH):
    if TRIPS_MONTH_URL:
        print("Trip file missing — downloading from TRIPS_MONTH_URL ...")
        try:
            download_file(TRIPS_MONTH_URL, RAW_TRIPS_PATH, expected='parquet')
        except Exception as e:
            raise RuntimeError(
                "Download failed. Check TRIPS_MONTH_URL and your internet connection. "
                f"Original error: {e}"
            )
    else:
        raise FileNotFoundError(
            f"Trip file not found at {RAW_TRIPS_PATH}. "
            "Set TRIPS_MONTH_URL (from the official TLC page) and re-run the download cell, "
            "or place the file manually in ./data/ and update RAW_TRIPS_PATH."
        )

# Load trips into Postgres
audit_df = load_trips(RAW_TRIPS_PATH)
audit_df.head()

fact_taxi_trips truncated (repeatable run).


,rows_in,dropped_missing_required,dropped_nonpositive_distance,dropped_negative_fare,dropped_negative_total,dropped_bad_time_order,dropped_absurd_duration,rows_out,rows_loaded,mode
0,3066766,0,45862,22037,113,3,88,2998663,2998663,parquet


### Notes
- Loaded trip data into `fact_taxi_trips` with repeatability controls.
- Applied validation rules during ingestion to prevent bad records entering analytics tables.
- Produced an audit log of rows in/out/loaded for operational transparency.

## 10. Post-Load Data Quality Checks

We verify:
- row count
- sanity ranges for distance, duration, totals
- null counts for key columns

In [11]:
with engine.begin() as conn:
    rowcount = conn.execute(text("SELECT COUNT(*) FROM fact_taxi_trips;")).fetchone()[0]

    sanity = conn.execute(text("""
        SELECT
          MIN(trip_distance), MAX(trip_distance),
          MIN(trip_duration_min), MAX(trip_duration_min),
          MIN(total_amount), MAX(total_amount)
        FROM fact_taxi_trips;
    """)).fetchone()

    nulls = pd.read_sql(text("""
        SELECT
          SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END) AS pickup_dt_nulls,
          SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END) AS pu_nulls,
          SUM(CASE WHEN do_location_id IS NULL THEN 1 ELSE 0 END) AS do_nulls
        FROM fact_taxi_trips;
    """), conn)

rowcount, sanity, nulls


(2998663,
 (0.01, 258928.15, 0.016666666666666666, 1439.8, 0.0, 1169.4),
    pickup_dt_nulls  pu_nulls  do_nulls
 0                0         0         0)

### Notes
- Confirmed the fact table is populated and queryable.
- Verified sanity ranges for key numeric fields after cleaning.
- Checked null rates for critical columns to detect ingestion regressions.

## 11. Analytics-Ready Queries (with Zone Enrichment)

Now that we have `dim_taxi_zones`, we can enrich:
- `pu_location_id` → pickup borough/zone
- `do_location_id` → dropoff borough/zone

This demonstrates **analytics readiness**, not just “data loaded”.

In [12]:
with engine.begin() as conn:
    peak_by_hour = pd.read_sql(text("""
        SELECT pickup_hour, COUNT(*) AS trips
        FROM fact_taxi_trips
        GROUP BY pickup_hour
        ORDER BY trips DESC;
    """), conn)

    top_pickup_zones = pd.read_sql(text("""
        SELECT z.borough, z.zone, COUNT(*) AS trips
        FROM fact_taxi_trips t
        JOIN dim_taxi_zones z ON t.pu_location_id = z.location_id
        GROUP BY z.borough, z.zone
        ORDER BY trips DESC
        LIMIT 15;
    """), conn)

    borough_flows = pd.read_sql(text("""
        SELECT pu.borough AS pickup_borough, dz.borough AS dropoff_borough, COUNT(*) AS trips
        FROM fact_taxi_trips t
        JOIN dim_taxi_zones pu ON t.pu_location_id = pu.location_id
        JOIN dim_taxi_zones dz ON t.do_location_id = dz.location_id
        GROUP BY pickup_borough, dropoff_borough
        ORDER BY trips DESC
        LIMIT 20;
    """), conn)

peak_by_hour.head(), top_pickup_zones.head(), borough_flows.head()


(   pickup_hour   trips
 0           18  211561
 1           17  205111
 2           15  192139
 3           16  191485
 4           19  188992,
      borough                          zone   trips
 0     Queens                   JFK Airport  154273
 1  Manhattan         Upper East Side South  146158
 2  Manhattan         Upper East Side North  136658
 3  Manhattan                Midtown Center  133390
 4  Manhattan  Penn Station/Madison Sq West  107636,
   pickup_borough dropoff_borough    trips
 0      Manhattan       Manhattan  2499453
 1         Queens       Manhattan   155489
 2      Manhattan          Queens    82507
 3      Manhattan        Brooklyn    63075
 4         Queens          Queens    61225)

### Notes
- Produced peak-hour demand results for operational insights.
- Enriched pickup zones for human-readable analytics.
- Generated borough-to-borough flows to demonstrate dimensional joins and query readiness.

## 12. Productionization notes (brief)

This notebook focuses on **clarity and correctness**. In production, I would:
- Orchestrate monthly loads with retries (Prefect/Airflow)
- Load via staging → swap for idempotency
- Run automated data quality checks (e.g., Great Expectations/dbt tests)
- Optimize bulk loads (COPY/warehouse-native ingestion) and create indexes after ingest


### Notes
- Mapped the notebook steps into an orchestrator-friendly task breakdown.
- Included `src/prefect_flow.py` as a minimal starting point for orchestration.

### Notes
- Documented how to evolve this demo into a production-grade pipeline.
- Highlighted practical improvements around storage formats, partitioning, and testing.

### Notes
- Recorded realistic limitations and clear next steps.
- Suggested production-standard improvements without overengineering the demo.

## 15. Summary

This project builds an end-to-end **data pipeline** on NYC TLC Yellow Taxi trip data:
- Ingests large raw files (Parquet/CSV) with basic validation
- Loads a star-schema design into PostgreSQL (`dim_taxi_zones`, `fact_taxi_trips`)
- Adds indexes for common analytics patterns
- Produces enriched SQL analytics (peak-hour demand, top pickup zones, borough flows)
